# MapBiomas Fogo – U-Net experimental – Brasil – MONITOR_01

**Campanha:** `brazil.monitor_01` | **Alvo:** área queimada | **Sensor:** Sentinel-2 (10 m) | **Periodicidade:** mensal | **Região:** Pantanal

Ponto de entrada fino. O código do experimento fica em `network/catalog/experiments/unet_pantanal/src` (branch `classification-unet`).
Usa os **mesmos mosaicos (M2) e amostras (M3)** do catálogo, no Cloud Storage, e não altera nada do catálogo.

# MapBiomas Fogo Monitor – Pipeline U-Net (Passos 0–7)

## Documentação e guia do usuário
Expanda esta seção para entender os passos, onde ficam os dados e como ler os resultados.

#### Introdução e contexto

Este notebook testa uma **U-Net** como alternativa à rede MLP por pixel do catálogo (M4/M5). A MLP olha **um pixel** por vez; a U-Net olha um **recorte de imagem (chip)** inteiro e aprende também a forma e a textura da cicatriz de fogo. Cada passo tem uma responsabilidade e **a sua própria escolha**, como no Monitor Brasil:

| Passo | O que faz | Escolhe | Saída (GCS `.../MONITOR_01/UNET_01/`) |
|---|---|---|---|
| **0** | Preparação: clona a branch, instala, login e configuração do projeto | – | – |
| **1** | Mosaicos: quais meses e bandas já foram exportados | – | resumo na tela |
| **2** | Escolhe o **treino** (amostras, bandas, índices, versão) ou reabre uma versão, confere as amostras e cria os chips (1 fogo / 0 não-fogo / 255 desconhecido) | amostras, bandas, índices, versão | `training_…/chips/`, `splits/`, `config.json` |
| **3** | Estatísticas de normalização (só chips de treino) | – | `stats/norm_stats.json` |
| **4** | Treina a U-Net com os chips de todos os meses de treino | épocas, batch | `models/` |
| **5** | Métricas de teste (limiar escolhido na validação) | – | `reports/metrics_*.json` |
| **6** | Classifica a região, um mapa por mês | versão do modelo e períodos de **classificação** | `training_…/predict/{região}_{período}_prob.tif` |
| **7** | Compara com a MLP (A bandas, B bandas + índices, C U-Net) | – | `reports/comparison_*.json` |

**Entradas:** mosaicos mensais (COGs do catálogo) e polígonos de fogo / não-fogo (CSVs do M3).
**Saídas:** modelo, métricas, mapas de probabilidade (int16 = probabilidade × 1000 + dayOfYear, igual ao M5) e relatórios, tudo em `UNET_01/`.

> **Nota:** o disco do Colab é **temporário**. Tudo o que importa fica no **Google Cloud Storage (GCS)**.

#### Períodos de treino x classificação

| Onde se escolhe | Para que serve | O que precisa existir |
|---|---|---|
| **Passo 2** – treino | arquivos de amostras escolhidos; os meses saem do nome deles e também entram nas métricas de teste | mosaico **e** amostras |
| **Passo 6** – classificação | meses a mapear com o modelo treinado | só o mosaico (com as bandas do modelo) |

Exemplo: treinar em `2026_10` e mapear `2026_09` e `2026_10`.

#### Como ler os resultados

Cada etapa imprime um resumo didático em português: 📋 inventário, ✂️ chips, 📏 normalização, 🧠 treino, 🎯 teste, 📊 comparação. Os marcadores ✅ (ok), ⚠️ (atenção), ❌ (problema) e 💡 (dica) aparecem nos diagnósticos.

#### Dicas

- Use **GPU (T4)** no Colab: *Ambiente de execução → Alterar tipo → GPU*. Na CPU o treino é muito lento.
- Não rode o treinador M4 do catálogo no mesmo ambiente: ele muda o TensorFlow para o modo v1.
- Mudou bandas, região ou amostras? Refaça os chips (`REFAZER_CHIPS = True` no passo 2).

## 🔧 Passo 0 — Preparação

In [ ]:
# 0.1 – Preparação do Colab (clona a branch, instala, ajusta os caminhos)
import sys, os, subprocess, shutil

BRANCH = "classification-unet"
REPO_DIR = "/content/brazil-fire"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/mapbiomas/brazil-fire.git", REPO_DIR], check=True)

EXP_DIR = os.path.join(REPO_DIR, "network/catalog/experiments/unet_pantanal")
subprocess.run(["pip", "install", "-q", "-r", os.path.join(EXP_DIR, "requirements.txt")], check=True)
sys.path.insert(0, os.path.join(EXP_DIR, "src"))
# descarta módulos unet_* já importados nesta sessão (senão o Python usa a versão antiga)
for _m in [m for m in sys.modules if m.startswith("unet_")]:
    del sys.modules[_m]

In [ ]:
# 0.2 – Configuração do projeto + login (períodos, bandas e opções são escolhidos em cada passo)
import unet_env
from unet_config import make_config, paths

UCFG = make_config(
    # projeto (onde estão os dados do catálogo)
    COUNTRY="brazil",
    BUCKET="mapbiomas-fire",
    GCS_PREFIX="sudamerica/brazil/CATALOG_01",
    CAMPAIGN="MONITOR_01",
    GEE_PROJECT="mapbiomas-brazil",
    # asset de regiões: region_name = bioma + "_r" + id (calculado, como no script do GEE)
    REGIONS_ASSET="users/geomapeamentoipam/AUXILIAR/regioes_biomas_col2",
    REGION_PROPERTY="region_name", REGION_NAME_FIELDS=["bioma", "id"], REGION_NAME_JOIN="_r",
    # mosaicos: LIBRARY_IMAGES/SENTINEL2/MONTHLY/MINNBR/{período}/COG/
    SENSOR="sentinel2",
    MOSAIC="minnbr",
    # --- períodos (formato 'AAAA_MM'): treino e classificação são listas separadas ---
    PERIODS=["2026_10"],             # TREINO: meses com mosaico E amostras (mais meses = mais exemplos)
    PREDICT_PERIODS=["2026_10"],     # CLASSIFICAÇÃO: meses a mapear (só precisam do mosaico; pode ser outro mês)
    # experimento
    EXPERIMENT="UNET_01",            # pasta no GCS, dentro da campanha (saídas)
    MODEL_NAME="unet_monitor_pantanal_v1",
    REGION="Pantanal_r1",            # region_name a classificar (se errar, o erro lista as opções)
    SAMPLE_REGION="Pantanal 60",     # valor da coluna `region` nos CSVs de amostras
    # Mosaico 2026_10 completo: blue, green, red, nir, swir1, swir2 + dayOfYear.
    BANDS=["blue", "green", "red", "nir", "swir1", "swir2"],
    # BANDS=["green", "swir2"],   # teste rápido, só 2 bandas
    INDICES=["nbr", "nbr2", "mirbi", "ndvi", "bai", "evi"],   # calculados das bandas; [] = sem índices
    VALID_BAND="dayOfYear",          # exportada: máscara de pixel válido + banda 2 da saída (None = sem ela)
    SAMPLE_GLOB="*{period}*.csv",    # CSVs M3 em MONITOR_01/LIBRARY_SAMPLES; ou SAMPLE_FILES=[...]
    CHIP=256, POINTS_PER_POLYGON=4, BLOCK_KM=25, SPLIT=(0.70, 0.15, 0.15),
    BATCH=8, EPOCHS=60, LR=1e-4, SEED=42,
    OVERLAP=16, INFER_TILE=512,
)
P = paths(UCFG)
unet_env.connect(UCFG)               # pop-up do Colab; GEE com a sua conta, GCS com a conta de serviço
print(f"Saídas:   gs://{P['bucket']}/{P['root']}/")
print(f"Amostras: gs://{P['bucket']}/{P['samples']}/")

## 📋 Passo 1 — Mosaicos disponíveis (só leitura)
Lista os meses que já têm mosaico no GCS e quais bandas foram exportadas. Não treina nada e não escolhe nada: serve para saber o que dá para usar no passo 2.

In [ ]:
from unet_ui import mostrar_mosaicos

mosaicos = mostrar_mosaicos(UCFG)

## ✂️ Passo 2 — Treino: escolha, amostras e chips
Aqui se escolhe o que entra no treino (2.1), se confere se mosaico e amostras estão certos (2.2) e se criam os chips de treino, validação e teste (2.3).

### 2.1 – Escolher o treino (versão)
Igual ao M4 da development. Aba **Novo treino**: escolha os **arquivos de amostras** (só aparecem os CSVs cujo mês tem mosaico; os períodos de treino saem do nome deles), as **bandas**, os **índices** e a **versão** (ID sugerido, shortname e comentário). Aba **Versões existentes**: reabre um treino já feito.

Cada versão tem a sua pasta `UNET_01/training_{id}_{shortname}_sentinel2/` com chips, estatísticas, modelo, métricas, mapas e o `config.json` que permite reabri-la.

In [ ]:
# 2.1a – Interface de escolha: novo treino (amostras, bandas, índices, versão) ou versão existente
from unet_config import paths
from unet_ui import run_ui_treino, apply_treino

ui_treino = run_ui_treino(UCFG)

In [ ]:
# 2.1b – Aplicar a escolha (rode de novo depois de mudar algo na interface acima).
# Novo treino: grava config.json na pasta da versão. Versão existente: reabre a configuração salva.
UCFG = apply_treino(ui_treino)
P = paths(UCFG)

### 2.2 – Conferir mosaico e amostras
Para cada mês de treino, confere se cada banda existe e está na mesma grade, lista os arquivos de amostras e resume bandas e polígonos por classe (o que observar).

In [ ]:
import json
from unet_io import MosaicReader, load_samples, list_sample_files
from unet_resumo import print_inventory

MOSTRAR_JSON = False   # True = imprime também o inventário bruto (JSON)

for period in UCFG["PERIODS"]:
    reader = MosaicReader(UCFG, period)
    inv = reader.inventory()
    sample_files = list_sample_files(UCFG, period)
    samples, report = load_samples(UCFG, period, reader.crs)
    print_inventory(UCFG, inv, sample_files, report)
    if MOSTRAR_JSON:
        print(json.dumps(inv, indent=2, default=str))
    reader.close()

### 2.3 – Criar os chips
Os blocos de `BLOCK_KM` são divididos em treino/validação/teste **antes** de fazer os chips. Um chip que cruza blocos de divisões diferentes é descartado. Rótulo: 1 fogo, 0 não-fogo, 255 fora dos polígonos ou nodata (a perda ignora). Onde os polígonos se sobrepõem, vale o fogo.

In [ ]:
from IPython.display import Image, display
from unet_chips import build_chips
from unet_io import local_dir
from unet_resumo import resumo_chips

# Cada versão nova começa sem chips, então False basta. True = apaga e refaz os chips
# DESTA versão (ex.: depois de mudar BLOCK_KM ou CHIP na configuração).
REFAZER_CHIPS = False

for period in UCFG["PERIODS"]:
    summary = build_chips(UCFG, period, overwrite=REFAZER_CHIPS)
    resumo_chips(UCFG, summary)
    if not summary.get("skipped"):
        png = os.path.join(local_dir("reports"), f"chips_preview_{period}.png")
        if os.path.exists(png):
            display(Image(png))

Confira a figura: vermelho = fogo, azul = não-fogo. Atenção a chips sobre lagoas e áreas alagadas (água escura parece cicatriz de fogo) e sobre nuvens ou sombras de nuvem que restaram no mosaico mensal.

## 📏 Passo 3 — Estatísticas de normalização (só chips de treino)

In [ ]:
from unet_chips import compute_stats
from unet_resumo import resumo_estatisticas

stats = compute_stats(UCFG)
resumo_estatisticas(stats)

## 🧠 Passo 4 — Treino da U-Net
Use GPU (T4): na CPU o treino é muito lento. O resumo mostra as curvas de aprendizado e um diagnóstico.

In [ ]:
# 4.1 – Opções do treino (épocas, batch) e aplicação
from unet_ui import run_ui_opcoes, apply_opcoes

ui_opcoes = run_ui_opcoes(UCFG)

In [ ]:
UCFG = apply_opcoes(ui_opcoes)

In [ ]:
from unet_model import train
from unet_resumo import resumo_treino, figura_treino

model, meta = train(UCFG)
resumo_treino(UCFG, meta, model)
figura_treino(meta)

## 🎯 Passo 5 — Métricas de teste
O limiar com melhor F1 é escolhido na **validação**; o teste é reportado nesse limiar e em 0,5.

In [ ]:
from unet_model import load_model
from unet_report import evaluate_unet
from unet_resumo import resumo_teste

model = load_model(UCFG)
unet_results = evaluate_unet(UCFG, model)
resumo_teste(unet_results)

## 🗺️ Passo 6 — Classificar a região inteira
Janela deslizante: cada janela lê `INFER_TILE + 2*OVERLAP` pixels e guarda só o centro,
para não aparecerem emendas. Saída no formato M5 (probabilidade × 1000 + dayOfYear,
nodata −9999), na mesma grade do mosaico.

In [ ]:
# 6.1 – Escolher a versão do modelo e os meses a classificar (só precisam de mosaico com as bandas do modelo)
from unet_ui import run_ui_classificar, apply_classificar

ui_classificar = run_ui_classificar(UCFG)

In [ ]:
UCFG = apply_classificar(ui_classificar)

In [ ]:
# 6.2 – Classificar com o modelo da versão escolhida
from unet_config import paths, region_slug
from unet_io import read_json
from unet_model import load_model
from unet_predict import predict_mosaic, gee_upload_command

P = paths(UCFG)
model = load_model(UCFG)              # modelo da versão escolhida em 6.1
stats = read_json(P["stats"])         # normalização da mesma versão
for period in UCFG["PREDICT_PERIODS"]:
    res = predict_mosaic(UCFG, period, model, stats)
    print(f"Mapa de {period} salvo em {res['gcs']}")
    print(f"   {res['valid_pixels']:,} pixels válidos | {res['burned_pixels_prob_gt_0']:,} com probabilidade > 0 "
          f"| {res['minutes']} min")
    print("   Para subir ao GEE (edite antes o caminho do asset):")
    print("   " + gee_upload_command(UCFG, period, f"projects/{UCFG['GEE_PROJECT']}/assets/FIRE/CATALOG_01/"
                                  f"{UCFG['CAMPAIGN']}/{UCFG['EXPERIMENT']}/{UCFG['MODEL_NAME']}/"
                                  f"{region_slug(UCFG)}_{period}_prob"))

## 📊 Passo 7 — Comparação com o MLP
Mesmos pixels de teste para todos os modelos. A = MLP com bandas, B = MLP com bandas + índices (pulado quando `INDICES` está vazio), C = U-Net.

In [ ]:
from IPython.display import Markdown
from unet_report import compare_with_mlp
from unet_resumo import resumo_comparacao

rows, table = compare_with_mlp(UCFG, unet_results)
resumo_comparacao(UCFG, rows)
display(Markdown(table))